# 01: Exploratory Data Analysis: Spanish Electricity Load

Narrative companion to `src/eda.py`. The dataset is 4 years of hourly national
electricity demand for Spain (ENTSO-E via the Kaggle dataset *Hourly energy
demand generation and weather*) merged with weather for the 5 largest cities.

**Question guiding the EDA:** what structure must a day-ahead forecasting model
capture? We look for (1) intraday shape, (2) weekly rhythm, (3) annual/weather
seasonality, and (4) autocorrelation that lag features can exploit.

> Run `python -m src.data_ingestion` first so `data/processed/` exists.

In [ ]:
import sys, json
from pathlib import Path
sys.path.append(str(Path.cwd().parent))

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from src.config import TARGET, LOCAL_TZ, REPORTS
from src.data_ingestion import load_processed

sns.set_theme(style="whitegrid")
df = load_processed()
local = df.index.tz_convert(LOCAL_TZ)
df = df.assign(hour=local.hour, weekday=local.weekday, month=local.month)
df.head()

## Data quality

Ingestion (`src/data_ingestion.py`) already handled the raw-data quirks:
duplicate weather rows, missing hours (re-indexed to a complete hourly range and
time-interpolated when the gap is ≤ 24 h), NaN targets, Kelvin→°C, and a stray
leading space in one city name. The exact counts for the current data are
logged to `reports/data_quality.json`:

In [ ]:
print(json.dumps(json.loads((REPORTS / 'data_quality.json').read_text()), indent=2))

## Intraday × weekly structure

The hour×weekday heatmap is the single most informative plot for load data.
Expect a Spanish double peak (midday + ~21:00 evening), a deep night trough,
and visibly lower weekend demand, evidence that `hour`, `weekday` and their
interaction matter.

In [ ]:
WEEKDAYS = ['Mon','Tue','Wed','Thu','Fri','Sat','Sun']
pivot = df.pivot_table(index='hour', columns='weekday', values=TARGET, aggfunc='mean')
pivot.columns = [WEEKDAYS[c] for c in pivot.columns]
plt.figure(figsize=(9, 7))
sns.heatmap(pivot, cmap='rocket_r', cbar_kws={'label': 'Mean load (MW)'})
plt.title('Mean load by hour and weekday')
plt.ylabel('Hour of day (local)');

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4.5))
for label, sub in (('Weekday', df[df.weekday < 5]), ('Weekend', df[df.weekday >= 5])):
    axes[0].plot(sub.groupby('hour')[TARGET].mean(), lw=2.5, label=label)
axes[0].set(title='Average intraday profile', xlabel='Hour (local)', ylabel='MW')
axes[0].legend()
how = df.weekday * 24 + df.hour
axes[1].plot(df.groupby(how)[TARGET].mean().values, lw=1.5)
axes[1].set_xticks(range(0, 168, 24), WEEKDAYS)
axes[1].set(title='Average hour-of-week profile')
fig.tight_layout()

## Trend and seasonality: STL on daily means

STL with a weekly period separates the strong 7-day cycle from the trend, which
itself carries the *annual* wave (winter + summer peaks). The remainder is what
models actually compete on.

In [ ]:
from statsmodels.tsa.seasonal import STL
daily = df[TARGET].resample('D').mean().interpolate()
res = STL(daily, period=7, robust=True).fit()
fig = res.plot()
fig.set_size_inches(11, 8)

## Autocorrelation

The ACF shows sharp spikes at multiples of 24 h and a dominant one at 168 h,
this is exactly why `lag_24h`, `lag_48h` and `lag_168h` are the backbone of the
feature set, and why a naive-24h forecast is already a serious benchmark.

In [ ]:
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
fig, axes = plt.subplots(2, 1, figsize=(12, 7))
plot_acf(df[TARGET].dropna(), lags=192, ax=axes[0])
plot_pacf(df[TARGET].dropna()[-5000:], lags=72, method='ywm', ax=axes[1])
fig.tight_layout()

## Load vs temperature: the U-shape

Demand rises at *both* temperature extremes: electric heating below ~15 °C and
air-conditioning above ~22 °C. A linear temperature term cannot represent this,
which motivates the squared-temperature feature (and the centred quadratic
exogenous term in SARIMAX).

In [ ]:
daily_all = df.resample('D').mean(numeric_only=True).dropna(subset=[TARGET, 'temp'])
plt.figure(figsize=(9, 6))
plt.scatter(daily_all['temp'], daily_all[TARGET], s=10, alpha=0.35)
plt.xlabel('Daily mean temperature across 5 cities (°C)')
plt.ylabel('Daily mean load (MW)')
plt.title('Heating + cooling demand → U-shape');

## Takeaways for modelling

1. **Three seasonalities** (24 h, 168 h, annual) → calendar + cyclical features.
2. **Strong autocorrelation at daily/weekly lags** → lag features that remain
   valid at a 24 h horizon (everything shifted ≥ 24 h).
3. **Non-linear temperature response** → `temp` + `temp²`.
4. **Weekend/holiday regime shift** → `is_weekend`, `is_holiday` (Spanish calendar).

Feature construction lives in `src/features.py`; modelling continues in
`02_modeling.ipynb`.